In [7]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
import random
import logging
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# Set up logging for debugging and tracking
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

def is_valid_image(url):
    """Check if the URL points to a valid image format (jpg, png, etc.)."""
    valid_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp')
    return url.lower().endswith(valid_extensions)

def get_image_urls(soup, base_url):
    """Extract and validate image URLs from the BeautifulSoup object."""
    image_urls = set()  # Use a set to avoid duplicate URLs
    for img_tag in soup.find_all("img"):  # Find all <img> tags
        img_url = img_tag.get("src") or img_tag.get("data-src")  # Check for src or data-src attribute
        if img_url:
            img_url = urljoin(base_url, img_url)  # Resolve relative URLs to absolute URLs
            if is_valid_image(img_url):  # Check if the image URL is valid
                image_urls.add(img_url)  # Add valid image URL to the set
    return image_urls

def save_image(img_url, save_dir, idx):
    """Download and save the image from the URL."""
    try:
        # Send HTTP request to fetch the image
        response = requests.get(img_url, stream=True)
        response.raise_for_status()  # Raise exception for HTTP errors
        # Construct the file path and name
        img_name = f"item_{idx}{os.path.splitext(urlparse(img_url).path)[-1]}"  # Get file extension from URL
        img_path = os.path.join(save_dir, img_name)
        # Save the image content to the specified file path
        with open(img_path, "wb") as f:
            for chunk in response.iter_content(1024):  # Download in chunks
                f.write(chunk)
        logger.info(f"Saved: {img_path}")  # Log the successful image saving
        return img_path
    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to save image from {img_url}: {e}")  # Log the error if saving fails
        return None

def setup_session():
    """Set up a requests session with retries and user-agent rotation."""
    session = requests.Session()  # Create a session to reuse connections
    # Define retry strategy for handling failed requests
    retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    session.mount('http://', HTTPAdapter(max_retries=retries))  # Retry for HTTP requests
    session.mount('https://', HTTPAdapter(max_retries=retries))  # Retry for HTTPS requests
    return session

def fetch_page_content(url, session):
    """Fetch the page content with proper headers and retry logic."""
    headers = {
        "User-Agent": random.choice([  # Randomly choose a User-Agent to simulate different browsers
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15",
            "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:88.0) Gecko/20100101 Firefox/88.0"
        ])
    }
    try:
        # Send GET request to fetch the page content
        response = session.get(url, headers=headers)
        response.raise_for_status()  # Raise exception for HTTP errors
        return response.content  # Return the page content if successful
    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to retrieve the webpage {url}: {e}")  # Log the error if request fails
        return None

def scrape_images(url, save_dir):
    """Main function to scrape and save images from a webpage."""
    session = setup_session()  # Set up session with retries
    page_content = fetch_page_content(url, session)  # Fetch page content
    if not page_content:
        return []  # Return empty list if page content retrieval fails

    soup = BeautifulSoup(page_content, "html.parser")  # Parse the HTML content using BeautifulSoup
    image_urls = get_image_urls(soup, url)  # Extract valid image URLs from the page

    # Create directory to save images if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    image_paths = []  # List to store paths of saved images
    for idx, img_url in enumerate(image_urls):  # Loop through each image URL
        img_path = save_image(img_url, save_dir, idx)  # Save image to disk
        if img_path:
            image_paths.append(img_path)  # Add the saved image path to the list
        time.sleep(random.uniform(1, 3))  # Random delay to mimic human behavior and avoid detection

    return image_paths

def main():
    """Main function to initiate the scraping process for multiple URLs."""
    ecommerce_urls = [
        # List of e-commerce URLs to scrape images from
        "https://www.flipkart.com/search?q=health%20supplements&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off",
        "https://www.flipkart.com/search?q=beauty-personal-care&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off",
        "https://www.flipkart.com/search?q=grocery&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off",
        "https://www.amazon.in/s?k=health+supplements&crid=FEYE81QM2CMU&sprefix=health+supplements+%2Caps%2C349&ref=nb_sb_noss_2",
        "https://www.amazon.in/s?k=grocery&crid=3BA55JGIDLUUF&sprefix=grocery%2Caps%2C420&ref=nb_sb_noss_2",
        "https://www.walmart.com/cp/health/976760",
        "https://www.walmart.com/cp/beauty/1085666",
        "https://www.walmart.com/search?q=groceries+",
        "https://www.ebay.com/sch/i.html?_from=R40&_trksid=p4439441.m570.l1313&_nkw=grocery&_sacat=0",
        "https://www.ebay.com/sch/i.html?_from=R40&_trksid=p4439441.m570.l1313&_nkw=health&_sacat=0",
        "https://www.ebay.com/sch/i.html?_from=R40&_trksid=p4439441.m570.l1313&_nkw=beauty&_sacat=0"
    ]

    for url in ecommerce_urls:  # Loop through each URL
        logger.info(f"Starting image scrape for: {url}")  # Log the start of scraping for a URL
        save_dir = os.path.join("scraped_images", urlparse(url).netloc)  # Define save directory based on domain
        image_paths = scrape_images(url, save_dir)  # Call the function to scrape images
        logger.info(f"Scraped {len(image_paths)} images from {url}")  # Log the number of images scraped

if __name__ == "__main__":
    main()  # Call the main function to start the script


Extracted: https://www.flipkart.com/santoor-wipro-skin-moisturizing-sandal-turmeric-bathing-bar-soap-soft-youthful/p/itmf872d73622ae3?pid=SOPF953TTAFG2DKM&lid=LSTSOPF953TTAFG2DKMI8UTOJ&marketplace=GROCERY&shopId=&iid=5868edbd-dfd2-4048-8e79-5d5ae8dae596.SOPF953TTAFG2DKM.SEARCH
Saved image 1 from URL: https://www.flipkart.com/santoor-wipro-skin-moisturizing-sandal-turmeric-bathing-bar-soap-soft-youthful/p/itmf872d73622ae3?pid=SOPF953TTAFG2DKM&lid=LSTSOPF953TTAFG2DKMI8UTOJ&marketplace=GROCERY&shopId=&iid=5868edbd-dfd2-4048-8e79-5d5ae8dae596.SOPF953TTAFG2DKM.SEARCH
Extracted: https://www.amazon.in/Park-Avenue-Signature-Voyage-235ml/dp/B01CSHNUFE/ref=sr_1_3_f3_0o_fs_sspa?crid=1DMO0ZLV9D0CW&dib=eyJ2IjoiMSJ9._JocfrqxbiEpGYAxeBya7m1DLU1u6OVoUfMbFnhd_qMBmj8tQJYXVts76Btza5eAQgH0wmaQ5DYuEmTjA1A9emPRXEcfGNxT7o9Ka5h5eVqpS1RgN_a99bgw8TY0v7eXfypnvWFGlDDCe8gcnYF-V36wmGOgU4Ka1JWJ1ybgeWHvhTWz2VflHyxe1X-6_6sypoIfi8h-80Y_pZDJKRaRzyCDZM5FGpT8OT296x6AdxOgM1r5Db7MjCTeJa0YmGF94Wq_qKh-thbKHawnTM5CcTH3IYjyFd70